<a href="https://colab.research.google.com/github/potato-yen/114-2-Programming-Language/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-generativeai gspread google-auth google-auth-oauthlib


In [2]:
# -*- coding: utf-8 -*-
from datetime import datetime
import json
import re

import gspread
import google.generativeai as legacy_genai  # 保留相容性，不直接使用
from google.colab import auth, userdata
from google.auth import default
from google import genai


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# Gemini 設定
api_key = userdata.get("gemini")
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-2.5-flash"


In [4]:
# Google Sheet 設定
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Etj3aomNgs1hroV3JwgRMNMVQ2_BIyKwChgtPRnYxic/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = [
    "日期",
    "類型",
    "科目",
    "分數",
    "整體表現",
    "待加強科目",
    "學習建議"
]

_auth_done = False
_gc = None
_ws = None


In [5]:
def authenticate_google_sheet():
    global _auth_done, _gc

    if not _auth_done:
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _auth_done = True

    return _gc


def get_worksheet():
    global _ws

    gc = authenticate_google_sheet()
    if _ws is None:
        sh = gc.open_by_url(SHEET_URL)
        _ws = sh.worksheet(WORKSHEET_NAME)

    return _ws


def ensure_sheet_columns(ws):
    values = ws.get_all_values()

    if not values:
        ws.append_row(REQUIRED_COLUMNS)
        print("已建立表頭。")
        return

    header = values[0]
    if header[:len(REQUIRED_COLUMNS)] != REQUIRED_COLUMNS:
        raise ValueError(
            "Google Sheet 的表頭格式與程式預期不一致。\n"
            f"目前表頭：{header}\n"
            f"預期表頭：{REQUIRED_COLUMNS}"
        )


In [6]:
def get_user_grades():
    """
    透過終端機輸入成績，直到使用者輸入 q 結束。
    每筆資料格式：
    [日期, 類型, 科目, 分數, 整體表現, 待加強科目, 學習建議]
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []

    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：").strip()
        if subject.lower() == "q":
            break

        if not subject:
            print("科目名稱不能空白，請重新輸入。")
            continue

        grade_raw = input(f"請輸入 {subject} 的成績（0~100）：").strip()

        try:
            grade = int(grade_raw)
        except ValueError:
            print("成績必須是整數，請重新輸入。")
            continue

        if not 0 <= grade <= 100:
            print("成績必須介於 0 到 100 之間，請重新輸入。")
            continue

        today = datetime.now().strftime("%Y-%m-%d")
        grades.append([today, "成績紀錄", subject, grade, "", "", ""])
        print(f"已記錄：日期={today}，科目={subject}，分數={grade}\n")

    return grades


In [7]:
def analyze_grades(grades):
    """
    根據輸入成績做基本統計，避免讓 AI 自己亂算。
    """
    if not grades:
        return None

    subject_scores = [{"subject": row[2], "score": row[3]} for row in grades]
    scores = [item["score"] for item in subject_scores]

    average_score = round(sum(scores) / len(scores), 1)
    highest_item = max(subject_scores, key=lambda x: x["score"])
    lowest_item = min(subject_scores, key=lambda x: x["score"])
    weak_subjects = [item["subject"] for item in subject_scores if item["score"] < 60]

    return {
        "count": len(scores),
        "average": average_score,
        "highest_subject": highest_item["subject"],
        "highest_score": highest_item["score"],
        "lowest_subject": lowest_item["subject"],
        "lowest_score": lowest_item["score"],
        "weak_subjects": weak_subjects
    }


def extract_json_object(text):
    """
    從模型輸出中抓取 JSON 物件。
    """
    if not text:
        raise ValueError("模型沒有回傳內容。")

    text = text.strip()

    # 先嘗試整段直接解析
    try:
        return json.loads(text)
    except Exception:
        pass

    # 再抓 ```json ... ``` 或一般大括號區塊
    match = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if not match:
        match = re.search(r"(\{.*\})", text, re.DOTALL)

    if not match:
        raise ValueError(f"找不到有效 JSON：{text}")

    return json.loads(match.group(1))


def get_ai_summary_structured(grades, stats):
    """
    呼叫 Gemini，要求輸出固定 JSON，避免內容雜亂。
    """
    grade_lines = []
    for row in grades:
        grade_lines.append(f"{row[2]}：{row[3]}分")

    prompt_text = f'''
你是一個學習成績整理助手。請根據以下成績資料，輸出「簡短、規整、可直接寫入 Google Sheet」的分析結果。

【成績資料】
{chr(10).join(grade_lines)}

【已計算統計】
- 科目數：{stats["count"]}
- 平均分：{stats["average"]}
- 最高分：{stats["highest_subject"]} {stats["highest_score"]}分
- 最低分：{stats["lowest_subject"]} {stats["lowest_score"]}分
- 低於60分科目：{", ".join(stats["weak_subjects"]) if stats["weak_subjects"] else "無"}

【輸出限制】
1. 只能輸出 JSON。
2. 不要加 markdown，不要加說明，不要加反引號。
3. JSON 格式固定如下：
{{
  "overall": "整體表現，25字內",
  "weak_subjects": "待加強科目整理，20字內；若無則寫無",
  "advice": "學習建議，40字內"
}}
4. 內容要精簡、自然、具體，不要空話。
5. 不要捏造不存在的資料，不要提到排名、標準差、班級比較。
'''.strip()

    print("\n--- 正在呼叫 AI 模型生成結構化摘要... ---")
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt_text
        )

        raw_text = response.text
        data = extract_json_object(raw_text)

        return {
            "overall": str(data.get("overall", "")).strip()[:25],
            "weak_subjects": str(data.get("weak_subjects", "")).strip()[:20],
            "advice": str(data.get("advice", "")).strip()[:40],
            "raw_text": raw_text
        }

    except Exception as e:
        print(f"呼叫 AI 或解析輸出時發生錯誤：{e}")
        return {
            "overall": "AI 摘要生成失敗",
            "weak_subjects": "無法分析",
            "advice": "請稍後再試",
            "raw_text": ""
        }


In [9]:
def build_summary_row(ai_result):
    today = datetime.now().strftime("%Y-%m-%d")
    return [
        today,
        "AI摘要",
        "",
        "",
        ai_result["overall"],
        ai_result["weak_subjects"],
        ai_result["advice"]
    ]


def main():
    """
    主程式流程：
    1. 驗證並連線 Google Sheet
    2. 取得使用者輸入的成績
    3. 先追加寫入原始成績
    4. 產生 AI 結構化摘要
    5. 再追加一列 AI 摘要
    """
    try:
        ws = get_worksheet()
        ensure_sheet_columns(ws)
        print("--- Google Sheet 連線成功。---")

        new_grades = get_user_grades()
        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        stats = analyze_grades(new_grades)

        # 先追加原始成績
        ws.append_rows(new_grades)
        print("\n--- 成績已成功追加到 Google Sheet。---")

        # 再追加 AI 摘要
        ai_result = get_ai_summary_structured(new_grades, stats)
        summary_row = build_summary_row(ai_result)
        ws.append_row(summary_row)
        print("--- AI 摘要已成功追加到 Google Sheet。---")

        print("\n--- 本次統計結果 ---")
        print(json.dumps(stats, ensure_ascii=False, indent=2))

        print("\n--- AI 摘要內容 ---")
        print(f"整體表現：{ai_result['overall']}")
        print(f"待加強科目：{ai_result['weak_subjects']}")
        print(f"學習建議：{ai_result['advice']}")

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")


if __name__ == "__main__":
    main()


--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：國文
請輸入 國文 的成績（0~100）：78
已記錄：日期=2026-04-01，科目=國文，分數=78

請輸入科目（或輸入 'q' 停止）：數學
請輸入 數學 的成績（0~100）：92
已記錄：日期=2026-04-01，科目=數學，分數=92

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功追加到 Google Sheet。---

--- 正在呼叫 AI 模型生成結構化摘要... ---
--- AI 摘要已成功追加到 Google Sheet。---

--- 本次統計結果 ---
{
  "count": 2,
  "average": 85.0,
  "highest_subject": "數學",
  "highest_score": 92,
  "lowest_subject": "國文",
  "lowest_score": 78,
  "weak_subjects": []
}

--- AI 摘要內容 ---
整體表現：表現良好，各科均達水準，數學成績尤其突出。
待加強科目：無
學習建議：保持數學優勢，國文可多加閱讀練習，穩固基礎，爭取更高分。
